In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


# Konfigurasi
CSV_PATH = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split.csv"
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/leakage/data_final_ready100_150.csv"
SEED = 42

# Gambar yang WAJIB masuk Test Set (analisis kualitatif skripsi)
MANDATORY_IMAGES = [
    "Data 2 Batik Hamparan Rintik (Tasik Madu).jpg",
    "Mekar Sari Data 6 Melati Deret.jpg",
    "Data 6 Batik Baronggung (Bunga Nirwana 2).jpg",
    "Data 13 Batik Kantil (Buntel).jpg",
    "Data 2 Batik Rahayu (Parang Barong).jpg",
    "Data 26 Batik Kantil (Kamalini Buana).jpg",
    "Data 4 Batik Soendari (Ikan Maskoki).jpg",
    "Data 26 Batik Rahayu (Sido Mukti Rahayu).jpg"
]

# Kuota Test Set per kelas (Total 65 citra Proporsional 20%)
TEST_QUOTAS = {
    'Malang': 74,
    'Lamongan': 44,
    'Trenggalek': 28,
    'Tulungagung': 4
}

def main():
    # 1. Load Data
    df = pd.read_csv(CSV_PATH)
    
    # Buat kolom baru, inisiasi dengan 'unused'
    df['new_split'] = 'unused'

    # 2. Amankan Gambar Mandatory ke Test Set
    # Cari indeks baris yang nama file-nya mengandung salah satu dari MANDATORY_IMAGES
    mandatory_mask = df['image_path'].apply(lambda path: any(img in path for img in MANDATORY_IMAGES))
    mandatory_indices = df[mandatory_mask].index.tolist()
    
    print(f"[*] Ditemukan {len(mandatory_indices)} dari {len(MANDATORY_IMAGES)} gambar mandatory.")
    
    # 3. Penuhi Kuota Test Set (Total 65)
    test_indices = list(mandatory_indices)
    
    for cls, quota in TEST_QUOTAS.items():
        # Hitung berapa mandatory image yang sudah masuk di kelas ini
        current_count = len(df.loc[mandatory_indices][df.loc[mandatory_indices, 'class'] == cls])
        needed = quota - current_count
        
        if needed > 0:
            # Ambil sisa data di kelas ini yang belum masuk test set
            available = df[(df['class'] == cls) & (~df.index.isin(test_indices))]
            # Sample acak sebanyak kekurangan
            sampled = available.sample(n=needed, random_state=SEED).index.tolist()
            test_indices.extend(sampled)
        elif needed < 0:
            print(f"WARNING: Gambar mandatory untuk kelas {cls} melebihi kuota test ({current_count} > {quota})!")

    # Tandai Test Set di DataFrame
    df.loc[test_indices, 'new_split'] = 'test'
    print(f"[*] Total Test Set Captioning: {len(test_indices)} citra (Murni Dataset A).")

    # 4. Reduksi Dataset B (Ambil 50 per kelas)
    dataset_a_classes = list(TEST_QUOTAS.keys())
    dataset_b_classes = [c for c in df['class'].unique() if c not in dataset_a_classes]
    
    dataset_b_sampled_indices = []
    for cls in dataset_b_classes:
        available = df[df['class'] == cls]
        # Ambil 50 citra per kelas
        sampled = available.sample(n=min(len(available), 100), random_state=SEED).index.tolist()
        dataset_b_sampled_indices.extend(sampled)
        
    print(f"[*] Total Dataset B tersisa setelah direduksi: {len(dataset_b_sampled_indices)} citra.")

    # 5. Gabungkan Sisa A dan Sampel B untuk Pool Pelatihan Classifier
    # Sisa A adalah kelas A yang BUKAN test set
    sisa_a_indices = df[(df['class'].isin(dataset_a_classes)) & (~df.index.isin(test_indices))].index.tolist()
    
    # Gabungkan indeks untuk pool (Train + Val)
    pool_indices = sisa_a_indices + dataset_b_sampled_indices
    pool_df = df.loc[pool_indices]
    
    print(f"[*] Total Data Latih Classifier (Sisa A + Sampel B): {len(pool_df)} citra.")

    # 6. Split 80:20 Stratified pada Pool
    train_idx, val_idx = train_test_split(
        pool_df.index, 
        test_size=0.20, 
        stratify=pool_df['class'], 
        random_state=SEED
    )
    
    df.loc[train_idx, 'new_split'] = 'train'
    df.loc[val_idx, 'new_split'] = 'val'

    # 7. Simpan dan Tampilkan Ringkasan
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[*] File berhasil disimpan sebagai: {OUTPUT_CSV}")
    
    print("\n--- RINGKASAN PEMBAGIAN (new_split) ---")
    print(df['new_split'].value_counts())
    
    print("\n--- DETAIL KELAS DI TEST SET ---")
    print(df[df['new_split'] == 'test']['class'].value_counts())
    
    print("\n--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---")
    print(df[df['new_split'] == 'val']['class'].value_counts().head(5), "...\n")

if __name__ == "__main__":
    main()

[*] Ditemukan 8 dari 8 gambar mandatory.
[*] Total Test Set Captioning: 150 citra (Murni Dataset A).
[*] Total Dataset B tersisa setelah direduksi: 2000 citra.
[*] Total Data Latih Classifier (Sisa A + Sampel B): 2177 citra.

[*] File berhasil disimpan sebagai: /mnt/extended-home/dzakaaufa/leakage/data_final_ready100_150.csv

--- RINGKASAN PEMBAGIAN (new_split) ---
new_split
train     1741
unused    1000
val        436
test       150
Name: count, dtype: int64

--- DETAIL KELAS DI TEST SET ---
class
Malang         74
Lamongan       44
Trenggalek     28
Tulungagung     4
Name: count, dtype: int64

--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---
class
betawi           20
singa_barong     20
buketan          20
bokor_kencono    20
sidoluhur        20
Name: count, dtype: int64 ...



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Konfigurasi
CSV_PATH = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split.csv"
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/leakage/data_final_ready1.csv"
SEED = 42

# Gambar yang WAJIB masuk Test Set (analisis kualitatif skripsi)
MANDATORY_IMAGES = [
    "Data 2 Batik Hamparan Rintik (Tasik Madu).jpg",
    "Mekar Sari Data 6 Melati Deret.jpg",
    "Data 6 Batik Baronggung (Bunga Nirwana 2).jpg",
    "Data 13 Batik Kantil (Buntel).jpg",
    "Data 2 Batik Rahayu (Parang Barong).jpg",
    "Data 26 Batik Kantil (Kamalini Buana).jpg",
    "Data 4 Batik Soendari (Ikan Maskoki).jpg",
    "Data 26 Batik Rahayu (Sido Mukti Rahayu).jpg"
]

# Kuota Test Set Baru (Total 150 citra, proporsional berdasarkan distribusi asli)
TEST_QUOTAS = {
    'Malang': 74,
    'Lamongan': 44,
    'Trenggalek': 28,
    'Tulungagung': 4
}

def main():
    # 1. Load Data
    df = pd.read_csv(CSV_PATH)
    
    # Buat kolom baru, inisiasi dengan 'unused'
    df['new_split'] = 'unused'

    # 2. Amankan Gambar Mandatory ke Test Set
    # Cari indeks baris yang nama file-nya mengandung salah satu dari MANDATORY_IMAGES
    mandatory_mask = df['image_path'].apply(lambda path: any(img in path for img in MANDATORY_IMAGES))
    mandatory_indices = df[mandatory_mask].index.tolist()
    
    print(f"[*] Ditemukan {len(mandatory_indices)} dari {len(MANDATORY_IMAGES)} gambar mandatory.")
    
    # 3. Penuhi Kuota Test Set (Total 150)
    test_indices = list(mandatory_indices)
    
    for cls, quota in TEST_QUOTAS.items():
        # Hitung berapa mandatory image yang sudah masuk di kelas ini
        current_count = len(df.loc[mandatory_indices][df.loc[mandatory_indices, 'class'] == cls])
        needed = quota - current_count
        
        if needed > 0:
            # Ambil sisa data di kelas ini yang belum masuk test set
            available = df[(df['class'] == cls) & (~df.index.isin(test_indices))]
            # Sample acak sebanyak kekurangan
            sampled = available.sample(n=needed, random_state=SEED).index.tolist()
            test_indices.extend(sampled)
        elif needed < 0:
            print(f"WARNING: Gambar mandatory untuk kelas {cls} melebihi kuota test ({current_count} > {quota})!")

    # Tandai Test Set di DataFrame
    df.loc[test_indices, 'new_split'] = 'test'
    print(f"[*] Total Test Set (Murni Bersih): {len(test_indices)} citra.")

    # 4. Reduksi Dataset B (Ambil Maksimal 50 per kelas)
    dataset_a_classes = list(TEST_QUOTAS.keys())
    dataset_b_classes = [c for c in df['class'].unique() if c not in dataset_a_classes]
    
    dataset_b_sampled_indices = []
    for cls in dataset_b_classes:
        available = df[df['class'] == cls]
        # Ambil maksimal 50 citra per kelas
        sampled = available.sample(n=min(len(available), 50), random_state=SEED).index.tolist()
        dataset_b_sampled_indices.extend(sampled)
        
    print(f"[*] Total Dataset B tersisa setelah direduksi (maks 50 per kelas): {len(dataset_b_sampled_indices)} citra.")

    # 5. Gabungkan Sisa A (196 citra) dan Sampel B untuk Pool Pelatihan Classifier
    # Sisa A adalah kelas A yang BUKAN bagian dari 150 test set
    sisa_a_indices = df[(df['class'].isin(dataset_a_classes)) & (~df.index.isin(test_indices))].index.tolist()
    print(f"[*] Total sisa data kelas A yang disebar kembali ke pool pelatihan: {len(sisa_a_indices)} citra.")
    
    # Gabungkan indeks untuk pool (Train + Val)
    pool_indices = sisa_a_indices + dataset_b_sampled_indices
    pool_df = df.loc[pool_indices]
    
    print(f"[*] Total Pool Data Pelatihan Classifier (196 Sisa A + Sampel B): {len(pool_df)} citra.")

    # 6. Split 80:20 Stratified pada Pool Pelatihan
    train_idx, val_idx = train_test_split(
        pool_df.index, 
        test_size=0.20, 
        stratify=pool_df['class'], 
        random_state=SEED
    )
    
    df.loc[train_idx, 'new_split'] = 'train'
    df.loc[val_idx, 'new_split'] = 'val'

    # 7. Simpan dan Tampilkan Ringkasan
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[SUKSES] File berhasil disimpan sebagai: {OUTPUT_CSV}")
    
    print("\n" + "="*50)
    print("RINGKASAN PEMBAGIAN DATASET BARU")
    print("="*50)
    print(df['new_split'].value_counts())
    
    print("\n--- DETAIL KELAS DI TEST SET (WAJIB 150) ---")
    print(df[df['new_split'] == 'test']['class'].value_counts())
    
    print("\n--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---")
    print(df[df['new_split'] == 'val']['class'].value_counts().head(5), "...\n")

if __name__ == "__main__":
    main()

[*] Ditemukan 8 dari 8 gambar mandatory.
[*] Total Test Set (Murni Bersih): 150 citra.
[*] Total Dataset B tersisa setelah direduksi (maks 50 per kelas): 1000 citra.
[*] Total sisa data kelas A yang disebar kembali ke pool pelatihan: 177 citra.
[*] Total Pool Data Pelatihan Classifier (196 Sisa A + Sampel B): 1177 citra.

[SUKSES] File berhasil disimpan sebagai: /mnt/extended-home/dzakaaufa/leakage/data_final_ready1.csv

RINGKASAN PEMBAGIAN DATASET BARU
new_split
unused    2000
train      941
val        236
test       150
Name: count, dtype: int64

--- DETAIL KELAS DI TEST SET (WAJIB 150) ---
class
Malang         74
Lamongan       44
Trenggalek     28
Tulungagung     4
Name: count, dtype: int64

--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---
class
Malang           18
Lamongan         10
buketan          10
bokor_kencono    10
srikaton         10
Name: count, dtype: int64 ...



In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
import torch
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
from tqdm import tqdm
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ================= KONFIGURASI =================
CSV_PATH = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split.csv"
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/leakage/data_final_ready_kmeans.csv"
SEED = 42
DINOV2_VERSION = "facebook/dinov2-base" # Bisa diganti ke dinov2-small jika VRAM terbatas

MANDATORY_IMAGES = [
    "Data 2 Batik Hamparan Rintik (Tasik Madu).jpg",
    "Mekar Sari Data 6 Melati Deret.jpg",
    "Data 6 Batik Baronggung (Bunga Nirwana 2).jpg",
    "Data 13 Batik Kantil (Buntel).jpg",
    "Data 2 Batik Rahayu (Parang Barong).jpg",
    "Data 26 Batik Kantil (Kamalini Buana).jpg",
    "Data 4 Batik Soendari (Ikan Maskoki).jpg",
    "Data 26 Batik Rahayu (Sido Mukti Rahayu).jpg"
]

TEST_QUOTAS = {
    'Malang': 74,
    'Lamongan': 44,
    'Trenggalek': 28,
    'Tulungagung': 4
}
# ===============================================

def get_dinov2_embeddings(image_paths, processor, model, device):
    """Fungsi ekstraksi fitur (CLS token) dari DINOv2 untuk sekumpulan gambar."""
    embeddings = []
    model.eval()
    with torch.no_grad():
        for path in tqdm(image_paths, desc="Ekstraksi Fitur", leave=False):
            try:
                img = Image.open(path).convert("RGB")
                inputs = processor(images=img, return_tensors="pt").to(device)
                outputs = model(**inputs)
                # Ambil CLS token (fitur global representasi gambar)
                cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()[0]
                embeddings.append(cls_embedding)
            except Exception as e:
                print(f"\n[Error] Gagal memproses {path}: {e}")
                # Jika error, isi dengan array nol agar dimensi tidak rusak
                embeddings.append(np.zeros(model.config.hidden_size))
    return np.array(embeddings)

def main():
    # 1. Load Data
    df = pd.read_csv(CSV_PATH)
    
    # Mencari kolom split asli secara dinamis (biasanya bernama 'split', 'new_split', atau sejenisnya)
    original_split_col = None
    for col in df.columns:
        if df[col].astype(str).str.lower().eq('test').any():
            original_split_col = col
            break
            
    # Inisialisasi kolom split baru untuk output nanti
    df['new_split'] = 'unused'

    # --- 2. KUNCI TEST SET ASLI (SAMA PERSIS TANPA RANDOMISASI) ---
    if original_split_col:
        print(f"[*] Menemukan kolom split asli: '{original_split_col}'")
        # Mengambil indeks baris yang di split asli ditandai sebagai 'test'
        test_indices = df[df[original_split_col].astype(str).str.lower() == 'test'].index.tolist()
        print(f"[*] Sukses mengunci {len(test_indices)} citra Test Set asli (Sama Persis).")
    else:
        print("[!] Peringatan: Kolom split berisi label 'test' tidak ditemukan di CSV asal!")
        print("[*] Menggunakan metode fallback (quota-based) dengan SEED untuk membangun Test Set...")
        # Fallback jika kolom tidak terdeteksi otomatis
        mandatory_mask = df['image_path'].apply(lambda path: any(img in path for img in MANDATORY_IMAGES))
        mandatory_indices = df[mandatory_mask].index.tolist()
        test_indices = list(mandatory_indices)
        for cls, quota in TEST_QUOTAS.items():
            current_count = len(df.loc[mandatory_indices][df.loc[mandatory_indices, 'class'] == cls])
            needed = quota - current_count
            if needed > 0:
                available = df[(df['class'] == cls) & (~df.index.isin(test_indices))]
                sampled = available.sample(n=needed, random_state=SEED).index.tolist()
                test_indices.extend(sampled)
                
    # Tandai Test Set di DataFrame baru
    df.loc[test_indices, 'new_split'] = 'test'
    
    # Verifikasi gambar mandatory tetap aman berada di test set
    mandatory_in_test = df.loc[test_indices, 'image_path'].apply(lambda path: any(img in path for img in MANDATORY_IMAGES)).sum()
    print(f"[*] Verifikasi: {mandatory_in_test} dari {len(MANDATORY_IMAGES)} gambar mandatory aman di dalam Test Set.")

    # --- 3. REDUKSI DATASET B MENGGUNAKAN FEATURE-SPACE CLUSTERING ---
    dataset_a_classes = list(TEST_QUOTAS.keys())
    dataset_b_classes = [c for c in df['class'].unique() if c not in dataset_a_classes]
    
    # Inisialisasi Model DINOv2 murni (Pre-trained)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[*] Memuat {DINOV2_VERSION} di {device.upper()} untuk Clustering Dataset B...")
    processor = AutoImageProcessor.from_pretrained(DINOV2_VERSION)
    dinov2_model = AutoModel.from_pretrained(DINOV2_VERSION).to(device)

    dataset_b_sampled_indices = []
    
    print("\n[*] Memulai seleksi Core-set untuk Dataset B:")
    for cls in dataset_b_classes:
        available = df[df['class'] == cls]
        
        if len(available) <= 50:
            print(f"  -> {cls}: {len(available)} citra (Diambil semua karena <= 50)")
            dataset_b_sampled_indices.extend(available.index.tolist())
        else:
            print(f"  -> {cls}: Reduksi {len(available)} -> 50 citra dengan K-Means")
            image_paths = available['image_path'].tolist()
            original_indices = available.index.tolist()
            
            # a. Ekstrak vektor fitur DINOv2
            features = get_dinov2_embeddings(image_paths, processor, dinov2_model, device)
            
            # b. Jalankan K-Means Clustering untuk mencari 50 kelompok visual
            kmeans = KMeans(n_clusters=50, random_state=SEED, n_init='auto')
            kmeans.fit(features)
            
            # c. Cari 1 gambar yang paling dekat dengan titik tengah (centroid) setiap kelompok
            closest_idx, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, features)
            
            # d. Konversi kembali ke indeks DataFrame asli
            best_50_indices = [original_indices[i] for i in closest_idx]
            dataset_b_sampled_indices.extend(best_50_indices)

    print(f"\n[*] Total Dataset B tersisa setelah K-Means: {len(dataset_b_sampled_indices)} citra.")

    # --- 4. GABUNGKAN SISA A & SAMPEL B, LALU SPLIT 80:20 ---
    # Sisa A dijamin pas 196 citra karena test_indices dikunci dari 150 data test asli
    sisa_a_indices = df[(df['class'].isin(dataset_a_classes)) & (~df.index.isin(test_indices))].index.tolist()
    print(f"[*] Total sisa data kelas A yang disebar kembali ke pool pelatihan: {len(sisa_a_indices)} citra (Harus 196).")
    
    pool_indices = sisa_a_indices + dataset_b_sampled_indices
    pool_df = df.loc[pool_indices]
    
    train_idx, val_idx = train_test_split(
        pool_df.index, 
        test_size=0.20, 
        stratify=pool_df['class'], 
        random_state=SEED
    )
    
    df.loc[train_idx, 'new_split'] = 'train'
    df.loc[val_idx, 'new_split'] = 'val'

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[SUKSES] File berhasil disimpan sebagai: {OUTPUT_CSV}")
    
    print("\n" + "="*50)
    print("RINGKASAN PEMBAGIAN DATASET BARU (K-MEANS)")
    print("="*50)
    print(df['new_split'].value_counts())
    
    print("\n--- DETAIL KELAS DI TEST SET (WAJIB 150) ---")
    print(df[df['new_split'] == 'test']['class'].value_counts())

if __name__ == "__main__":
    main()

/mnt/extended-home/dzakaaufa/capenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Menemukan kolom split asli: 'split'
[*] Sukses mengunci 150 citra Test Set asli (Sama Persis).
[*] Verifikasi: 8 dari 8 gambar mandatory aman di dalam Test Set.

[*] Memuat facebook/dinov2-base di CUDA untuk Clustering Dataset B...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



[*] Memulai seleksi Core-set untuk Dataset B:
  -> betawi: Reduksi 150 -> 50 citra dengan K-Means


  -> wirasat: Reduksi 150 -> 50 citra dengan K-Means


...

  -> sekarjagad: Reduksi 150 -> 50 citra dengan K-Means


  -> sidomukti: Reduksi 150 -> 50 citra dengan K-Means


  -> mega_mendung: Reduksi 150 -> 50 citra dengan K-Means


  -> jlamprang: Reduksi 150 -> 50 citra dengan K-Means



[*] Total Dataset B tersisa setelah K-Means: 1000 citra.
[*] Total sisa data kelas A yang disebar kembali ke pool pelatihan: 177 citra (Harus 196).

[SUKSES] File berhasil disimpan sebagai: /mnt/extended-home/dzakaaufa/leakage/data_final_ready_kmeans.csv

RINGKASAN PEMBAGIAN DATASET BARU (K-MEANS)
new_split
unused    2000
train      941
val        236
test       150
Name: count, dtype: int64

--- DETAIL KELAS DI TEST SET (WAJIB 150) ---
class
Malang         74
Lamongan       43
Trenggalek     28
Tulungagung     5
Name: count, dtype: int64


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Konfigurasi
CSV_PATH = "/mnt/extended-home/dzakaaufa/dataset/caption/data_mix_inject_split.csv"
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/leakage/model150_65/data150.csv"
SEED = 42

# Gambar yang WAJIB masuk Test Set (analisis kualitatif skripsi)
MANDATORY_IMAGES = [
    "Data 2 Batik Hamparan Rintik (Tasik Madu).jpg",
    "Mekar Sari Data 6 Melati Deret.jpg",
    "Data 6 Batik Baronggung (Bunga Nirwana 2).jpg",
    "Data 13 Batik Kantil (Buntel).jpg",
    "Data 2 Batik Rahayu (Parang Barong).jpg",
    "Data 26 Batik Kantil (Kamalini Buana).jpg",
    "Data 4 Batik Soendari (Ikan Maskoki).jpg",
    "Data 26 Batik Rahayu (Sido Mukti Rahayu).jpg"
]

# Kuota Test Set per kelas (Total 65 citra Proporsional 20%)
TEST_QUOTAS = {
    'Malang': 32,
    'Lamongan': 19,
    'Trenggalek': 12,
    'Tulungagung': 2
}

def main():
    # 1. Load Data
    df = pd.read_csv(CSV_PATH)
    
    # Buat kolom baru, inisiasi dengan 'unused'
    df['new_split'] = 'unused'

    # 2. Amankan Gambar Mandatory ke Test Set
    # Cari indeks baris yang nama file-nya mengandung salah satu dari MANDATORY_IMAGES
    mandatory_mask = df['image_path'].apply(lambda path: any(img in path for img in MANDATORY_IMAGES))
    mandatory_indices = df[mandatory_mask].index.tolist()
    
    print(f"[*] Ditemukan {len(mandatory_indices)} dari {len(MANDATORY_IMAGES)} gambar mandatory.")
    
    # 3. Penuhi Kuota Test Set (Total 65)
    test_indices = list(mandatory_indices)
    
    for cls, quota in TEST_QUOTAS.items():
        # Hitung berapa mandatory image yang sudah masuk di kelas ini
        current_count = len(df.loc[mandatory_indices][df.loc[mandatory_indices, 'class'] == cls])
        needed = quota - current_count
        
        if needed > 0:
            # Ambil sisa data di kelas ini yang belum masuk test set
            available = df[(df['class'] == cls) & (~df.index.isin(test_indices))]
            # Sample acak sebanyak kekurangan
            sampled = available.sample(n=needed, random_state=SEED).index.tolist()
            test_indices.extend(sampled)
        elif needed < 0:
            print(f"WARNING: Gambar mandatory untuk kelas {cls} melebihi kuota test ({current_count} > {quota})!")

    # Tandai Test Set di DataFrame
    df.loc[test_indices, 'new_split'] = 'test'
    print(f"[*] Total Test Set Captioning: {len(test_indices)} citra (Murni Dataset A).")

    # 4. Reduksi Dataset B (Ambil 50 per kelas)
    dataset_a_classes = list(TEST_QUOTAS.keys())
    dataset_b_classes = [c for c in df['class'].unique() if c not in dataset_a_classes]
    
    dataset_b_sampled_indices = []
    for cls in dataset_b_classes:
        available = df[df['class'] == cls]
        # Ambil 50 citra per kelas
        sampled = available.sample(n=min(len(available), 150), random_state=SEED).index.tolist()
        dataset_b_sampled_indices.extend(sampled)
        
    print(f"[*] Total Dataset B tersisa setelah direduksi: {len(dataset_b_sampled_indices)} citra.")

    # 5. Gabungkan Sisa A dan Sampel B untuk Pool Pelatihan Classifier
    # Sisa A adalah kelas A yang BUKAN test set
    sisa_a_indices = df[(df['class'].isin(dataset_a_classes)) & (~df.index.isin(test_indices))].index.tolist()
    
    # Gabungkan indeks untuk pool (Train + Val)
    pool_indices = sisa_a_indices + dataset_b_sampled_indices
    pool_df = df.loc[pool_indices]
    
    print(f"[*] Total Data Latih Classifier (Sisa A + Sampel B): {len(pool_df)} citra.")

    # 6. Split 80:20 Stratified pada Pool
    train_idx, val_idx = train_test_split(
        pool_df.index, 
        test_size=0.20, 
        stratify=pool_df['class'], 
        random_state=SEED
    )
    
    df.loc[train_idx, 'new_split'] = 'train'
    df.loc[val_idx, 'new_split'] = 'val'

    # 7. Simpan dan Tampilkan Ringkasan
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n[*] File berhasil disimpan sebagai: {OUTPUT_CSV}")
    
    print("\n--- RINGKASAN PEMBAGIAN (new_split) ---")
    print(df['new_split'].value_counts())
    
    print("\n--- DETAIL KELAS DI TEST SET ---")
    print(df[df['new_split'] == 'test']['class'].value_counts())
    
    print("\n--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---")
    print(df[df['new_split'] == 'val']['class'].value_counts().head(5), "...\n")

if __name__ == "__main__":
    main()

[*] Ditemukan 8 dari 8 gambar mandatory.
[*] Total Test Set Captioning: 65 citra (Murni Dataset A).
[*] Total Dataset B tersisa setelah direduksi: 3000 citra.
[*] Total Data Latih Classifier (Sisa A + Sampel B): 3262 citra.

[*] File berhasil disimpan sebagai: /mnt/extended-home/dzakaaufa/leakage/model150_65/data150.csv

--- RINGKASAN PEMBAGIAN (new_split) ---
new_split
train    2609
val       653
test       65
Name: count, dtype: int64

--- DETAIL KELAS DI TEST SET ---
class
Malang         32
Lamongan       19
Trenggalek     12
Tulungagung     2
Name: count, dtype: int64

--- DETAIL KELAS DI VAL SET (Khusus DINOv2) ---
class
betawi        30
liong         30
tujuh_rupa    30
dayak         30
parang        30
Name: count, dtype: int64 ...

